In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import random
import glob
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
 
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.transforms.functional as TF

In [3]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
 
# ── Paths (Kaggle layout) ─────────────────────────────────────────────────────
BASE_DIR        = "/kaggle/input/competitions/plant-leaves-super-resolution-challenge"         
TRAIN_HR_DIR    = os.path.join(BASE_DIR, "train_High_Resolution")
TRAIN_LR_DIR    = os.path.join(BASE_DIR, "train_Low_Resolution")
TEST_LR_DIR     = os.path.join(BASE_DIR, "test_Low_Resolution")
VGG_WEIGHTS     = os.path.join(BASE_DIR, "vgg19_weights.pth")
OUTPUT_DIR      = "/kaggle/working"

In [ ]:
# ── Hyper-parameters ──────────────────────────────────────────────────────────
SCALE_FACTOR    = 4          # 32 → 128
LR_SIZE         = 32
HR_SIZE         = 128
BATCH_SIZE      = 32
NUM_EPOCHS      = 150
LR_G            = 2e-4       # generator learning rate
LR_D            = 1e-4       # discriminator learning rate
LAMBDA_PIXEL    = 100.0      # weight for L1 pixel loss  ← most critical
LAMBDA_PERCEP   = 10.0       # weight for perceptual loss
LAMBDA_ADV      = 1.0        # weight for adversarial loss
VAL_SPLIT       = 0.1        # 10 % of training data for validation
NUM_RES_BLOCKS  = 16         # residual blocks in generator
# OLD
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# NEW — explicitly use T4 and enable mixed precision
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print(f"Device : {DEVICE}")
print(f"Epochs : {NUM_EPOCHS}  |  Batch : {BATCH_SIZE}")

# Mixed precision — uses Tensor Cores on T4 for ~2x speedup
USE_AMP = torch.cuda.is_available()
scaler_G = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_D = torch.cuda.amp.GradScaler(enabled=USE_AMP)
 

In [5]:
# ── CELL 2: Dataset Class ─────────────────────────────────────────────────────
class AgriVisionDataset(Dataset):
    """
    Paired LR / HR dataset.
    Augmentations are applied identically to both images via a shared random
    state so spatial correspondence is preserved.
    """
    def __init__(self, lr_dir, hr_dir, augment=True):
        self.lr_paths  = sorted(glob.glob(os.path.join(lr_dir, "*.png")))
        self.hr_paths  = sorted(glob.glob(os.path.join(hr_dir, "*.png")))
        assert len(self.lr_paths) == len(self.hr_paths), \
            "Mismatch between LR and HR image counts!"
        self.augment = augment
 
    def __len__(self):
        return len(self.lr_paths)
 
    def __getitem__(self, idx):
        lr_img = Image.open(self.lr_paths[idx]).convert("RGB")
        hr_img = Image.open(self.hr_paths[idx]).convert("RGB")
 
        # ── Shared augmentation ───────────────────────────────────────────────
        if self.augment:
            # Horizontal flip
            if random.random() > 0.5:
                lr_img = TF.hflip(lr_img)
                hr_img = TF.hflip(hr_img)
            # Vertical flip
            if random.random() > 0.5:
                lr_img = TF.vflip(lr_img)
                hr_img = TF.vflip(hr_img)
            # 90-degree rotation (0 / 90 / 180 / 270)
            angle = random.choice([0, 90, 180, 270])
            if angle != 0:
                lr_img = TF.rotate(lr_img, angle)
                hr_img = TF.rotate(hr_img, angle)
 
        # ── To tensor & normalise to [-1, 1] ─────────────────────────────────
        lr_tensor = TF.to_tensor(lr_img)          # [0, 1]
        hr_tensor = TF.to_tensor(hr_img)
        lr_tensor = lr_tensor * 2.0 - 1.0         # [-1, 1]
        hr_tensor = hr_tensor * 2.0 - 1.0
 
        return lr_tensor, hr_tensor
 
 
class AgriVisionTestDataset(Dataset):
    """Test-only dataset (no HR available)."""
    def __init__(self, lr_dir):
        self.lr_paths = sorted(glob.glob(os.path.join(lr_dir, "*.png")))
 
    def __len__(self):
        return len(self.lr_paths)
 
    def __getitem__(self, idx):
        fname  = os.path.basename(self.lr_paths[idx])
        lr_img = Image.open(self.lr_paths[idx]).convert("RGB")
        lr_t   = TF.to_tensor(lr_img) * 2.0 - 1.0
        return fname, lr_t

In [6]:
full_dataset = AgriVisionDataset(TRAIN_LR_DIR, TRAIN_HR_DIR, augment=True)
 
val_size   = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)
# Turn off augmentation for validation split
val_ds.dataset.augment = False
 
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_ds      = AgriVisionTestDataset(TEST_LR_DIR)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False,
                          num_workers=2, pin_memory=True)
 
print(f"Train samples : {train_size}  |  Val samples : {val_size}  |  Test samples : {len(test_ds)}")

Train samples : 1478  |  Val samples : 164  |  Test samples : 495


In [ ]:
def denorm(t):
    """[-1,1] tensor → [0,1] numpy (H,W,C)."""
    return ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()
 
lr_sample, hr_sample = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i in range(6):
    axes[0, i].imshow(denorm(lr_sample[i]))
    axes[0, i].set_title("LR 32×32"); axes[0, i].axis("off")
    axes[1, i].imshow(denorm(hr_sample[i]))
    axes[1, i].set_title("HR 128×128"); axes[1, i].axis("off")
plt.suptitle("Sample LR / HR Pairs", fontsize=14)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "sample_pairs.png"), dpi=80)
plt.show()
print("Visualisation saved.")
 

In [8]:
 
class ResidualBlock(nn.Module):
    """Classic ResBlock with two 3×3 convolutions + skip connection."""
    def __init__(self, channels=64):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(channels),
            nn.PReLU(),
            nn.Conv2d(channels, channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(channels),
        )
 
    def forward(self, x):
        return x + self.block(x)
 
 
class UpsampleBlock(nn.Module):
    """Sub-pixel convolution for 2× upsampling."""
    def __init__(self, channels=64):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels * 4, 3, 1, 1)
        self.ps   = nn.PixelShuffle(2)      # channels*4 → channels, H×2, W×2
        self.act  = nn.PReLU()
 
    def forward(self, x):
        return self.act(self.ps(self.conv(x)))
 
 
class Generator(nn.Module):
    """
    SRGAN-style generator:
      Conv → [ResBlocks] → Conv → [2× Upsample blocks] → Conv (tanh)
    For 4× SR we stack two UpsampleBlocks (2×2 = 4×).
    """
    def __init__(self, in_channels=3, feat=64, num_res=NUM_RES_BLOCKS):
        super().__init__()
        # Initial feature extraction
        self.initial = nn.Sequential(
            nn.Conv2d(in_channels, feat, 9, 1, 4),
            nn.PReLU()
        )
        # Residual trunk
        res_blocks = [ResidualBlock(feat) for _ in range(num_res)]
        self.res_blocks = nn.Sequential(*res_blocks)
        # Post-residual conv
        self.post_res = nn.Sequential(
            nn.Conv2d(feat, feat, 3, 1, 1, bias=False),
            nn.BatchNorm2d(feat)
        )
        # Two 2× upsample blocks → 4× total
        self.upsample = nn.Sequential(
            UpsampleBlock(feat),
            UpsampleBlock(feat),
        )
        # Output: tanh maps to [-1,1]
        self.output = nn.Sequential(
            nn.Conv2d(feat, in_channels, 9, 1, 4),
            nn.Tanh()
        )
        self._init_weights()
 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
 
    def forward(self, x):
        feat   = self.initial(x)
        res    = self.res_blocks(feat)
        res    = self.post_res(res) + feat   # global residual skip
        up     = self.upsample(res)
        return self.output(up)
 

In [9]:
# ── CELL 6: Discriminator — PatchGAN ─────────────────────────────────────────
 
class DiscBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, use_bn=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=not use_bn)]
        if use_bn:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)
 
    def forward(self, x):
        return self.block(x)
 
 
class Discriminator(nn.Module):
    """
    PatchGAN discriminator.
    Receives concatenated [LR_upsampled | HR_candidate] so it is conditional.
    Input channels = 6 (3 LR + 3 HR/fake).
    Outputs a patch-level real/fake map.
    """
    def __init__(self, in_channels=6):
        super().__init__()
        self.model = nn.Sequential(
            DiscBlock(in_channels, 64,  stride=1, use_bn=False),
            DiscBlock(64,  128, stride=2),
            DiscBlock(128, 128, stride=1),
            DiscBlock(128, 256, stride=2),
            DiscBlock(256, 256, stride=1),
            DiscBlock(256, 512, stride=2),
            DiscBlock(512, 512, stride=1),
            nn.Conv2d(512, 1, 3, 1, 1)      # no sigmoid — use BCEWithLogitsLoss
        )
        self._init_weights()
 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02)
                nn.init.zeros_(m.bias)
 
    def forward(self, lr_up, hr):
        x = torch.cat([lr_up, hr], dim=1)
        return self.model(x)
 


In [10]:
# ── CELL 7: VGG-19 Perceptual Loss (uses provided weights only) ───────────────
 
class VGG19Features(nn.Module):
    """
    Extracts features from VGG-19 relu3_4 layer for perceptual loss.
    Loads from the competition-provided .pth file — no internet needed.
    """
    def __init__(self, weights_path):
        super().__init__()
        import torchvision.models as models
        # Build architecture shell (no pretrained download)
        vgg = models.vgg19(pretrained=False)
        # Load competition-provided weights
        state = torch.load(weights_path, map_location="cpu")
        vgg.load_state_dict(state)
        # Use features up to relu3_4 (index 18)
        self.features = nn.Sequential(*list(vgg.features.children())[:18])
        for param in self.parameters():
            param.requires_grad = False
        # ImageNet normalisation for VGG input
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))
 
    def forward(self, x):
        # x is in [-1,1]; convert to [0,1] then ImageNet normalise
        x = (x + 1.0) / 2.0
        x = (x - self.mean) / self.std
        return self.features(x)
 
 
class PerceptualLoss(nn.Module):
    def __init__(self, weights_path):
        super().__init__()
        self.vgg  = VGG19Features(weights_path)
        self.loss = nn.L1Loss()
 
    def forward(self, fake, real):
        return self.loss(self.vgg(fake), self.vgg(real))

In [11]:
# ── CELL 8: Combined generator loss ──────────────────────────────────────────
 
class GeneratorLoss(nn.Module):
    def __init__(self, weights_path=None, use_perceptual=True):
        super().__init__()
        self.pixel_loss    = nn.L1Loss()
        self.adv_loss      = nn.BCEWithLogitsLoss()
        self.use_perceptual = use_perceptual and (weights_path is not None) \
                              and os.path.exists(weights_path)
        if self.use_perceptual:
            self.perceptual = PerceptualLoss(weights_path)
            print("Perceptual loss: ENABLED (VGG-19 weights loaded)")
        else:
            print("Perceptual loss: DISABLED (weights not found or not requested)")
 
    def forward(self, fake_hr, real_hr, disc_fake):
        # L1 pixel loss — highest weight, key for MAE leaderboard
        l_pixel = self.pixel_loss(fake_hr, real_hr) * LAMBDA_PIXEL
 
        # Adversarial: Generator wants D to output 1 for fake images
        real_labels = torch.ones_like(disc_fake)
        l_adv = self.adv_loss(disc_fake, real_labels) * LAMBDA_ADV
 
        # Perceptual (optional)
        l_percep = torch.tensor(0.0, device=fake_hr.device)
        if self.use_perceptual:
            l_percep = self.perceptual(fake_hr, real_hr) * LAMBDA_PERCEP
 
        total = l_pixel + l_adv + l_percep
        return total, l_pixel, l_adv, l_percep
 
 

In [ ]:
# CELL 9: Initialise Models, Optimisers, Schedulers (T4 + AMP + DataParallel)
# =============================================================================
 
# ── GPU Info ──────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i} : {props.name}  |  VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("No GPU found — running on CPU")
 
# ── Mixed Precision Scalers ───────────────────────────────────────────────────
USE_AMP  = torch.cuda.is_available()
scaler_G = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_D = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print(f"Mixed Precision (AMP): {'ENABLED' if USE_AMP else 'DISABLED'}")
 
# ── Build Models ──────────────────────────────────────────────────────────────
G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)
 
# ── DataParallel for T4 x2 ────────────────────────────────────────────────────
if torch.cuda.device_count() > 1:
    print(f"DataParallel: ENABLED across {torch.cuda.device_count()} GPUs")
    G = nn.DataParallel(G)
    D = nn.DataParallel(D)
else:
    print("DataParallel: DISABLED (single GPU or CPU)")
 
# ── Optimisers ────────────────────────────────────────────────────────────────
# Access parameters via .module when DataParallel is used
G_params = G.module.parameters() if isinstance(G, nn.DataParallel) else G.parameters()
D_params = D.module.parameters() if isinstance(D, nn.DataParallel) else D.parameters()
 
opt_G = optim.Adam(G_params, lr=LR_G, betas=(0.9, 0.999))
opt_D = optim.Adam(D_params, lr=LR_D, betas=(0.9, 0.999))
 
# ── LR Schedulers ─────────────────────────────────────────────────────────────
sched_G = optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=NUM_EPOCHS, eta_min=1e-6)
sched_D = optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=NUM_EPOCHS, eta_min=1e-6)
 
# ── Loss Functions ────────────────────────────────────────────────────────────
criterion_G = GeneratorLoss(VGG_WEIGHTS, use_perceptual=True).to(DEVICE)
criterion_D = nn.BCEWithLogitsLoss()
 
# ── Bicubic upsampler for Discriminator conditioning ─────────────────────────
upsample_lr = nn.Upsample(scale_factor=SCALE_FACTOR, mode='bicubic',
                           align_corners=False).to(DEVICE)
 
# ── Parameter Count ───────────────────────────────────────────────────────────
G_core = G.module if isinstance(G, nn.DataParallel) else G
D_core = D.module if isinstance(D, nn.DataParallel) else D
param_count_G = sum(p.numel() for p in G_core.parameters() if p.requires_grad)
param_count_D = sum(p.numel() for p in D_core.parameters() if p.requires_grad)
print(f"Generator params    : {param_count_G:,}")
print(f"Discriminator params: {param_count_D:,}")

In [13]:

# ── CELL 10: Validation MAE helper ───────────────────────────────────────────
 
def compute_val_mae(generator, loader):
    """Compute pixel-level MAE on validation set (same metric as leaderboard)."""
    generator.eval()
    total_mae = 0.0
    n = 0
    with torch.no_grad():
        for lr, hr in loader:
            lr, hr = lr.to(DEVICE), hr.to(DEVICE)
            fake   = generator(lr)
            # denorm to [0,255]
            fake_255 = ((fake.clamp(-1,1) + 1) / 2 * 255).round()
            hr_255   = ((hr.clamp(-1,1)   + 1) / 2 * 255).round()
            total_mae += (fake_255 - hr_255).abs().mean().item() * lr.size(0)
            n += lr.size(0)
    generator.train()
    return total_mae / n


In [14]:
# CELL 11: Training Loop (T4 + AMP + DataParallel)
# =============================================================================
 
history = {"epoch": [], "loss_G": [], "loss_D": [], "val_mae": []}
best_val_mae   = float("inf")
best_ckpt_path = os.path.join(OUTPUT_DIR, "best_generator.pth")
 
print(f"\n{'='*65}")
print(f" Starting Training  |  {NUM_EPOCHS} epochs  |  Device: {DEVICE}")
print(f" AMP: {USE_AMP}  |  GPUs: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")
print(f"{'='*65}\n")
 
for epoch in range(1, NUM_EPOCHS + 1):
    G.train()
    D.train()
    epoch_loss_G = 0.0
    epoch_loss_D = 0.0
 
    for batch_idx, (lr, hr) in enumerate(train_loader):
        lr    = lr.to(DEVICE)       # [B, 3,  32,  32] in [-1, 1]
        hr    = hr.to(DEVICE)       # [B, 3, 128, 128] in [-1, 1]
        lr_up = upsample_lr(lr)     # [B, 3, 128, 128] — bicubic, for D input
 
        # ── 1. Train Discriminator ────────────────────────────────────────────
        opt_D.zero_grad()
 
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            fake_hr    = G(lr).detach()          # detach so G gets no gradient here
            real_preds = D(lr_up, hr)
            fake_preds = D(lr_up, fake_hr)
 
            # Label smoothing for training stability
            real_labels = torch.ones_like(real_preds)  * 0.9
            fake_labels = torch.zeros_like(fake_preds) + 0.1
 
            loss_D_real = criterion_D(real_preds, real_labels)
            loss_D_fake = criterion_D(fake_preds, fake_labels)
            loss_D      = (loss_D_real + loss_D_fake) * 0.5
 
        scaler_D.scale(loss_D).backward()
        scaler_D.step(opt_D)
        scaler_D.update()
 
        # ── 2. Train Generator ────────────────────────────────────────────────
        opt_G.zero_grad()
 
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            fake_hr   = G(lr)                    # fresh forward pass with gradients
            disc_fake = D(lr_up, fake_hr)
            loss_G, l_pix, l_adv, l_per = criterion_G(fake_hr, hr, disc_fake)
 
        scaler_G.scale(loss_G).backward()
        # Unscale before gradient clipping so clip threshold is meaningful
        scaler_G.unscale_(opt_G)
        G_core = G.module if isinstance(G, nn.DataParallel) else G
        nn.utils.clip_grad_norm_(G_core.parameters(), max_norm=1.0)
        scaler_G.step(opt_G)
        scaler_G.update()
 
        epoch_loss_G += loss_G.item()
        epoch_loss_D += loss_D.item()
 
    # ── Step LR Schedulers ────────────────────────────────────────────────────
    sched_G.step()
    sched_D.step()
 
    # ── Validation MAE ────────────────────────────────────────────────────────
    val_mae = compute_val_mae(G, val_loader)
 
    avg_G = epoch_loss_G / len(train_loader)
    avg_D = epoch_loss_D / len(train_loader)
 
    history["epoch"].append(epoch)
    history["loss_G"].append(avg_G)
    history["loss_D"].append(avg_D)
    history["val_mae"].append(val_mae)
 
    # ── Save Best Checkpoint ──────────────────────────────────────────────────
    # Always save .module.state_dict() when DataParallel is used
    # so the checkpoint can be loaded on a single GPU during inference
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        model_to_save = G.module if isinstance(G, nn.DataParallel) else G
        torch.save(model_to_save.state_dict(), best_ckpt_path)
        tag = " ← BEST"
    else:
        tag = ""
 
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:>3}/{NUM_EPOCHS}]  "
              f"G: {avg_G:.4f}  D: {avg_D:.4f}  "
              f"Val MAE: {val_mae:.4f}  "
              f"LR_G: {sched_G.get_last_lr()[0]:.2e}"
              f"{tag}")
 
print(f"\nTraining complete.  Best Val MAE: {best_val_mae:.4f}")
print(f"Best checkpoint saved at: {best_ckpt_path}")



 Starting Training  |  150 epochs  |  Device: cuda
 AMP: True  |  GPUs: 2



/tmp/ipykernel_23/2594167608.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_23/2594167608.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoch [  1/150]  G: 46.2742  D: 0.6219  Val MAE: 26.0609  LR_G: 2.00e-04 ← BEST
Epoch [  5/150]  G: 29.0439  D: 0.3305  Val MAE: 20.0728  LR_G: 1.99e-04 ← BEST
Epoch [ 10/150]  G: 27.1698  D: 0.3263  Val MAE: 18.9044  LR_G: 1.98e-04
Epoch [ 15/150]  G: 26.5185  D: 0.3259  Val MAE: 19.5058  LR_G: 1.95e-04
Epoch [ 20/150]  G: 26.2363  D: 0.3257  Val MAE: 18.2231  LR_G: 1.91e-04 ← BEST
Epoch [ 25/150]  G: 25.9114  D: 0.3256  Val MAE: 19.2218  LR_G: 1.87e-04
Epoch [ 30/150]  G: 25.7715  D: 0.3255  Val MAE: 18.5969  LR_G: 1.81e-04
Epoch [ 35/150]  G: 25.5959  D: 0.3254  Val MAE: 18.7600  LR_G: 1.74e-04
Epoch [ 40/150]  G: 25.4547  D: 0.3254  Val MAE: 18.1712  LR_G: 1.67e-04
Epoch [ 45/150]  G: 25.3544  D: 0.3254  Val MAE: 17.8020  LR_G: 1.59e-04
Epoch [ 50/150]  G: 25.8483  D: 0.5430  Val MAE: 19.6627  LR_G: 1.50e-04
Epoch [ 55/150]  G: 26.1016  D: 0.3627  Val MAE: 17.8052  LR_G: 1.41e-04
Epoch [ 60/150]  G: 26.5751  D: 0.4284  Val MAE: 17.6995  LR_G: 1.31e-04
Epoch [ 65/150]  G: 25.5921  D

In [ ]:
 
# ── CELL 12: Plot training curves ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history["epoch"], history["loss_G"], label="G Loss")
axes[0].plot(history["epoch"], history["loss_D"], label="D Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Generator & Discriminator Loss"); axes[0].legend()
 
axes[1].plot(history["epoch"], history["val_mae"], color="orange")
axes[1].axhline(y=17.3529644, color="red", linestyle="--", label="Baseline (17.35)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MAE")
axes[1].set_title("Validation MAE"); axes[1].legend()
 
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=100)
plt.show()


In [16]:
G_infer = Generator().to(DEVICE)
G_infer.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))
# No DataParallel on inference — single GPU is fine and simpler
G_infer.eval()

Generator(
  (initial): Sequential(
    (0): Conv2d(3, 64, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
    (1): PReLU(num_parameters=1)
  )
  (res_blocks): Sequential(
    (0): ResidualBlock(
      (block): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): PReLU(num_parameters=1)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): ResidualBlock(
      (block): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): PReLU(num_parameters=1)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias

In [17]:
# ── CELL 14: Test-Time Augmentation (TTA) helper ─────────────────────────────
# Average predictions from 8 augmented versions (4 rotations × 2 flips)
# This reduces variance and improves MAE
 
def tta_predict(model, lr_tensor):
    """
    lr_tensor: [1, 3, 32, 32] on DEVICE.
    Returns averaged prediction [1, 3, 128, 128].
    """
    preds = []
    for flip in [False, True]:
        for k in [0, 1, 2, 3]:          # 0°, 90°, 180°, 270°
            aug = lr_tensor.clone()
            if flip:
                aug = torch.flip(aug, dims=[3])    # horizontal flip
            if k > 0:
                aug = torch.rot90(aug, k=k, dims=[2, 3])
            with torch.no_grad():
                out = model(aug)
            # Reverse transforms
            if k > 0:
                out = torch.rot90(out, k=(4 - k), dims=[2, 3])
            if flip:
                out = torch.flip(out, dims=[3])
            preds.append(out)
    return torch.stack(preds).mean(dim=0)

In [18]:
# ── CELL 15: Generate predictions & build submission CSV ─────────────────────
print("Running inference on test set...")
 
rows = []
for fnames, lr_batch in test_loader:
    lr_batch = lr_batch.to(DEVICE)     # [B, 3, 32, 32]
 
    for i in range(lr_batch.size(0)):
        lr_single = lr_batch[i:i+1]    # [1, 3, 32, 32]
 
        # TTA prediction
        fake_hr = tta_predict(G_infer, lr_single)   # [1, 3, 128, 128]
 
        # Denormalise [-1,1] → [0,255] integers
        fake_hr_255 = ((fake_hr.clamp(-1, 1) + 1) / 2 * 255)
        fake_hr_255 = fake_hr_255.round().clamp(0, 255).to(torch.uint8)
 
        # Convert to HWC numpy and flatten in row-major RGB order
        img_np  = fake_hr_255.squeeze(0).permute(1, 2, 0).cpu().numpy()  # (128,128,3)
        flat    = img_np.flatten().astype(np.uint8)                        # 49,152 values
 
        # Verify shape
        assert flat.shape[0] == 49152, f"Expected 49152 values, got {flat.shape[0]}"
 
        pixel_str = " ".join(map(str, flat))
        rows.append({"Id": fnames[i], "Pixels": pixel_str})
 
submission_df = pd.DataFrame(rows, columns=["Id", "Pixels"])
submission_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission_df.to_csv(submission_path, index=False)
print(f"Submission saved: {submission_path}")
print(f"Total rows       : {len(submission_df)}")
print(submission_df.head(3))

Running inference on test set...
Submission saved: /kaggle/working/submission.csv
Total rows       : 495
                         Id                                             Pixels
0  agrivision_test_0000.png  149 141 145 151 146 147 153 147 150 154 146 15...
1  agrivision_test_0001.png  44 49 40 37 40 31 32 31 25 27 26 20 23 23 17 2...
2  agrivision_test_0002.png  140 135 132 140 138 134 142 140 138 142 141 14...


In [19]:
# ── CELL 16: Validate submission format ──────────────────────────────────────
print("\n── Submission Format Check ──")
df_check = pd.read_csv(submission_path)
assert list(df_check.columns) == ["Id", "Pixels"], "Column names wrong!"
assert len(df_check) == len(test_ds), f"Row count mismatch: {len(df_check)} vs {len(test_ds)}"
 
for idx, row in df_check.iterrows():
    vals = row["Pixels"].split(" ")
    assert len(vals) == 49152, f"Row {idx}: expected 49152 values, got {len(vals)}"
    ints = [int(v) for v in vals]
    assert all(0 <= v <= 255 for v in ints), f"Row {idx}: pixel out of [0,255] range!"
 
print("✓ All rows have exactly 49,152 space-separated integers in [0, 255].")
print("✓ Format check passed — ready to submit!")



── Submission Format Check ──
✓ All rows have exactly 49,152 space-separated integers in [0, 255].
✓ Format check passed — ready to submit!


In [ ]:
# ── CELL 17: Visual sanity check ─────────────────────────────────────────────
print("\nVisual sanity check on first 4 test images...")
G_infer.eval()
fnames_vis, lr_vis = next(iter(test_loader))
lr_vis = lr_vis.to(DEVICE)
 
with torch.no_grad():
    fake_vis = G_infer(lr_vis[:4])
 
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0, i].imshow(denorm(lr_vis[i]))
    axes[0, i].set_title(f"LR Input\n{fnames_vis[i]}", fontsize=8)
    axes[0, i].axis("off")
    axes[1, i].imshow(denorm(fake_vis[i]))
    axes[1, i].set_title("SR Output 128×128", fontsize=8)
    axes[1, i].axis("off")
 
plt.suptitle("Test Set — Low-Resolution Input vs Super-Resolved Output", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "test_predictions.png"), dpi=100)
plt.show()
 
print("\n✓ Pipeline complete.")
print(f"  Best checkpoint : {best_ckpt_path}")
print(f"  Submission CSV  : {submission_path}")
print(f"  Best Val MAE    : {best_val_mae:.4f}  (baseline: 17.3529644)")